# Notebook 02 — TextBraTS Dataset Validation & Quality Assessment

---

## Objectives

Runs integrity checks on the report inventory produced in Notebook 01, mirroring the rigor of the CV module's Notebook 02 (dataset validation for the MRI volumes).

Checks covered:
- Duplicate patient IDs / duplicate report content
- Empty reports
- Text encoding problems
- Word-count outliers
- Vocabulary size and medical-term coverage

> All checks here run on the **full, unsplit** inventory (369 patients) — this is intentional and safe. Nothing computed in this notebook is a fitted model parameter; it is purely descriptive QA, exactly like the CV module's pre-split EDA notebooks. The patient-level train/validation/test split only happens in Notebook 03, and no statistic from this notebook is reused as a preprocessing parameter after that point.


In [1]:
from pathlib import Path
import json
import hashlib
import re
from collections import Counter

import numpy as np
import pandas as pd

## 1. Setup & Load Inventory


In [2]:
RESULTS_DIR = Path("/kaggle/input/datasets/mariammohamed1095/reports/reports/nlp")

dataset_df = pd.read_csv(
    RESULTS_DIR / "textbrats_dataset_inventory.csv"
)

print(f"Dataset shape: {dataset_df.shape}")

dataset_df.head()

Dataset shape: (369, 10)


,patient_id,txt_exists,npy_exists,txt_path,npy_path,report,characters,words,lines,embedding_shape
0,BraTS20_Training_001,True,True,/kaggle/input/datasets/mariammohamed1095/textb...,/kaggle/input/datasets/mariammohamed1095/textb...,The lesion area is in the right frontal and pa...,586,91,1,"(1, 128, 768)"
1,BraTS20_Training_002,True,True,/kaggle/input/datasets/mariammohamed1095/textb...,/kaggle/input/datasets/mariammohamed1095/textb...,"The lesion area is in the right frontal lobe, ...",888,139,1,"(1, 128, 768)"
2,BraTS20_Training_003,True,True,/kaggle/input/datasets/mariammohamed1095/textb...,/kaggle/input/datasets/mariammohamed1095/textb...,"The lesion area is in the left frontal lobe, p...",804,128,1,"(1, 128, 768)"
3,BraTS20_Training_004,True,True,/kaggle/input/datasets/mariammohamed1095/textb...,/kaggle/input/datasets/mariammohamed1095/textb...,The lesion area is in the left temporal lobe a...,449,71,1,"(1, 128, 768)"
4,BraTS20_Training_005,True,True,/kaggle/input/datasets/mariammohamed1095/textb...,/kaggle/input/datasets/mariammohamed1095/textb...,The lesion area is in the left parietal lobe r...,606,94,1,"(1, 128, 768)"


## 2. Duplicate Patient ID Check


In [3]:
print("=" * 60)

print("Patients :", len(dataset_df))

print("Unique Patients :", dataset_df.patient_id.nunique())

print("Duplicate Patient IDs :",
      dataset_df.patient_id.duplicated().sum())

Patients : 369
Unique Patients : 369
Duplicate Patient IDs : 0


## 3. Missing Values


In [4]:
missing_summary = dataset_df.isna().sum()

display(missing_summary)

patient_id         0
txt_exists         0
npy_exists         0
txt_path           0
npy_path           0
report             0
characters         0
words              0
lines              0
embedding_shape    0
dtype: int64

## 4. Duplicate Report Content


In [5]:
duplicate_reports = dataset_df[
    dataset_df.report.duplicated(keep=False)
]

print("Duplicate reports:", len(duplicate_reports))

Duplicate reports: 0


## 5. Empty Reports


In [6]:
empty_reports = dataset_df[
    dataset_df.report.str.strip() == ""
]

print("Empty reports:", len(empty_reports))

Empty reports: 0


## 6. Content-Hash Duplicate Check

A stricter duplicate check than Section 4 — hashes the full report text so even reports identical except for invisible whitespace differences would still be caught.


In [7]:
dataset_df["report_hash"] = dataset_df.report.apply(
    lambda x: hashlib.sha256(
        x.encode("utf-8")
    ).hexdigest()
)

print(
    "Unique hashes:",
    dataset_df.report_hash.nunique()
)

print(
    "Duplicate hashes:",
    dataset_df.report_hash.duplicated().sum()
)

Unique hashes: 369
Duplicate hashes: 0


## 7. Encoding Validation


In [8]:
def encoding_problem(text):

    try:

        text.encode("utf-8")

        return False

    except UnicodeEncodeError:

        return True

### 7.1 Run Encoding Check


In [9]:
dataset_df["encoding_problem"] = dataset_df.report.apply(
    encoding_problem
)

print(
    "Encoding problems:",
    dataset_df.encoding_problem.sum()
)

Encoding problems: 0


## 8. Word-Count Outliers (IQR Method)


In [10]:
Q1 = dataset_df.words.quantile(.25)

Q3 = dataset_df.words.quantile(.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR

upper = Q3 + 1.5 * IQR

outliers = dataset_df[
    (dataset_df.words < lower)
    |
    (dataset_df.words > upper)
]

print(outliers.shape)

outliers[
    [
        "patient_id",
        "words"
    ]
]

(3, 12)


,patient_id,words
136,BraTS20_Training_137,153
212,BraTS20_Training_213,146
313,BraTS20_Training_314,156


## 9. Vocabulary Analysis


In [11]:
all_words = []

for report in dataset_df.report:

    all_words.extend(
        report.lower().split()
    )

vocab = Counter(all_words)

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 1003


### 9.1 Most Common Words


In [12]:
pd.DataFrame(

    vocab.most_common(30),

    columns=[
        "Word",
        "Frequency"
    ]

)

,Word,Frequency
0,the,3439
1,and,1717
2,is,1603
3,in,1328
4,of,1301
5,with,1230
6,high,756
7,left,736
8,parietal,730
9,signal,704


### 9.2 Medical-Term Coverage

Spot-checks whether expected clinical vocabulary (lesion, edema, necrosis, anatomical lobes...) actually appears in the corpus — a sanity signal that these are genuine radiology reports, not placeholder text.


In [13]:
medical_terms = [

"lesion",

"edema",

"necrosis",

"compression",

"signal",

"signals",

"mixed",

"frontal",

"parietal",

"temporal",

"occipital",

"ventricular",

"lobe"

]

coverage = []

for term in medical_terms:

    coverage.append({

        "term": term,

        "count": vocab.get(term, 0)

    })

coverage = pd.DataFrame(coverage)

coverage

,term,count
0,lesion,614
1,edema,456
2,necrosis,391
3,compression,457
4,signal,704
5,signals,279
6,mixed,353
7,frontal,639
8,parietal,730
9,temporal,315


### 9.3 Lexical Diversity


In [14]:
total_words = sum(vocab.values())

lexical_diversity = len(vocab) / total_words

print(

    f"Vocabulary Size : {len(vocab)}"

)

print(

    f"Total Words : {total_words}"

)

print(

    f"Lexical Diversity : {lexical_diversity:.4f}"

)

Vocabulary Size : 1003
Total Words : 35734
Lexical Diversity : 0.0281


## 10. Integrity Checklist


In [15]:
checks = {

    "Missing TXT":
    (~dataset_df.txt_exists).sum() == 0,

    "Missing NPY":
    (~dataset_df.npy_exists).sum() == 0,

    "Duplicate IDs":
    dataset_df.patient_id.duplicated().sum() == 0,

    "Duplicate Reports":
    len(duplicate_reports) == 0,

    "Duplicate Hashes":
    dataset_df.report_hash.duplicated().sum() == 0,

    "Empty Reports":
    len(empty_reports) == 0,

    "Encoding":
    dataset_df.encoding_problem.sum() == 0,

}

### 10.1 Summary Table & Score


In [16]:
summary = pd.DataFrame({

    "Check": checks.keys(),

    "Passed": checks.values()

})

display(summary)

score = (

    sum(checks.values())

    /

    len(checks)

) * 100

print(

    f"\nDataset Integrity Score : {score:.1f}/100"

)

,Check,Passed
0,Missing TXT,True
1,Missing NPY,True
2,Duplicate IDs,True
3,Duplicate Reports,True
4,Duplicate Hashes,True
5,Empty Reports,True
6,Encoding,True



Dataset Integrity Score : 100.0/100


## 11. Save Validation Report


In [17]:
RESULTS_DIR = Path("/kaggle/working/reports/nlp/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

### 11.1 Save CSV


In [18]:
summary.to_csv(

    RESULTS_DIR /
    "dataset_validation_summary.csv",

    index=False

)

### 11.2 Save JSON


In [19]:
checks_python = {
    k: bool(v)
    for k, v in checks.items()
}

with open(
    RESULTS_DIR / "dataset_validation.json",
    "w",
) as f:

    json.dump(
        {
            "patients": int(len(dataset_df)),
            "vocabulary_size": int(len(vocab)),
            "integrity_score": float(score),
            "checks": checks_python,
        },
        f,
        indent=4,
    )

print("Validation report saved.")

Validation report saved.
